# Category Signature Comparison
Aggregates per-page leaf signatures and spatial descriptors by structural category (tagged in `05_page_categorization.ipynb`), and produces comparison plots and persisted summary artifacts under `data/xrf/output/comparison/`.

In [9]:
import json
from pathlib import Path

import numpy as np

from xrf.config import Xrf_Comparison_Config, Leaf_Signature_Config
from xrf.signatures.leaf_signature import Leaf_Signature_Extractor
from xrf.spatial.spatial_analyzer import Spatial_Analyzer
from xrf.comparison.category_registry import Category_Registry
from xrf.comparison.category_signatures import Category_Signature_Aggregator
from xrf.comparison.spatial_comparison import Category_Spatial_Comparator
from xrf.visualization.xrf_plots import (
    Plot_Category_Signature_Bars,
    Plot_Category_Signature_Radar,
    Build_Category_Montage,
)

comparison_config = Xrf_Comparison_Config()
sig_config = Leaf_Signature_Config()

In [11]:
# Project directory and data paths
PROJECT_DIR = Path.cwd()

# Find project root directory
while not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "xrf"
OUTPUT_DATA_DIR = DATA_DIR / "output"
PROCESSED_DATA_DIR = OUTPUT_DATA_DIR / "processed"
FIGURES_DIR = OUTPUT_DATA_DIR / "figures"
COMPARISON_DIR = OUTPUT_DATA_DIR / "comparison"
MONTAGES_DIR = COMPARISON_DIR / "montages"

COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
MONTAGES_DIR.mkdir(parents=True, exist_ok=True)

## Load per-page signatures, categories, and spatial descriptors
Only tagged pages (structural_category present) are included in the comparison — untagged pages are skipped with a warning.

In [12]:
Page_Signatures = {}
Page_Categories = {}
Page_Spatial_Descriptors = {}

for Meta_Path in sorted(PROCESSED_DATA_DIR.glob("page_*_meta.json")):
    Page_Id = Meta_Path.stem.replace("_meta", "")
    Category = Category_Registry.Load_Page_Category(Meta_Path)

    if Category is None:
        print(f"Skipping {Page_Id}: not yet tagged (run 05_page_categorization.ipynb).")
        continue

    with open(Meta_Path, "r") as Meta_File:
        Meta = json.load(Meta_File)
    Optimal_K = Meta["optimal_k"]

    Labels = np.load(PROCESSED_DATA_DIR / f"{Page_Id}_labels.npy")
    Class_Map = np.load(PROCESSED_DATA_DIR / f"{Page_Id}_class_map.npy")

    Page_Signatures[Page_Id] = Leaf_Signature_Extractor.Compute_Abundances(
        Labels, Num_Classes=Optimal_K
    )
    Page_Categories[Page_Id] = Category

    Page_Spatial_Descriptors[Page_Id] = {
        Class_K: Spatial_Analyzer.Extract_Spatial_Descriptors(
            Class_Map, Target_Class=Class_K, Min_Size=sig_config.Min_Region_Size
        )
        for Class_K in range(Optimal_K)
    }

print(f"Loaded {len(Page_Signatures)} tagged page(s) across "
      f"{len(set(Page_Categories.values()))} category/ies.")

Loaded 1 tagged page(s) across 1 category/ies.


## Aggregate signatures by category

In [13]:
Category_Signatures = Category_Signature_Aggregator.Aggregate_By_Category(
    Page_Signatures, Page_Categories
)
Category_Spread = Category_Signature_Aggregator.Compute_Category_Spread(
    Page_Signatures, Page_Categories
)

for Category, Signature in Category_Signatures.items():
    np.save(COMPARISON_DIR / f"category_signature_{Category}.npy", Signature)
    np.save(COMPARISON_DIR / f"category_spread_{Category}.npy", Category_Spread[Category])
    print(f"{Category}: mean={np.round(Signature, 3)}, mad={np.round(Category_Spread[Category], 3)}")

text_only: mean=[0.094 0.061 0.137 0.208 0.076 0.087 0.214 0.122], mad=[0. 0. 0. 0. 0. 0. 0. 0.]


## Aggregate spatial descriptors by category

In [14]:
Category_Region_Stats = Category_Spatial_Comparator.Aggregate_Region_Stats(
    Page_Spatial_Descriptors, Page_Categories
)
Category_Region_Stats

{'text_only': {0: {'mean_region_count': 196.0,
   'mean_region_size': 19.760204081632654},
  1: {'mean_region_count': 129.0, 'mean_region_size': 18.728682170542637},
  2: {'mean_region_count': 270.0, 'mean_region_size': 90.87407407407407},
  3: {'mean_region_count': 983.0, 'mean_region_size': 45.63377416073245},
  4: {'mean_region_count': 202.0, 'mean_region_size': 14.178217821782178},
  5: {'mean_region_count': 223.0, 'mean_region_size': 19.91031390134529},
  6: {'mean_region_count': 532.0, 'mean_region_size': 100.25375939849624},
  7: {'mean_region_count': 307.0, 'mean_region_size': 22.29641693811075}}}

## Plots

In [15]:
Plot_Category_Signature_Bars(
    Category_Signatures, Category_Spread, COMPARISON_DIR / "category_signature_bars.png"
)
Plot_Category_Signature_Radar(
    Category_Signatures, COMPARISON_DIR / "category_signature_radar.png"
)

[Xrf_Plots] saved category signature bar chart to c:\Users\gabri\Documento\Mitacs\research_ct\data\xrf\output\comparison\category_signature_bars.png
[Xrf_Plots] saved category signature radar overlay to c:\Users\gabri\Documento\Mitacs\research_ct\data\xrf\output\comparison\category_signature_radar.png


## Cluster montage (illustrative)
`Build_Category_Montage` assembles existing `Cluster_k_Visual.png` files for visual review. With only a handful of processed pages so far, this montage draws from the shared book-level cluster visuals in `data/xrf/output/figures/` rather than a true per-category set — revisit once per-category cluster exports exist.

In [16]:
Cluster_Visual_Paths = sorted(FIGURES_DIR.glob("Cluster_*_Visual.png"))

if Cluster_Visual_Paths:
    for Category in Category_Signatures:
        Build_Category_Montage(
            Cluster_Visual_Paths,
            MONTAGES_DIR / f"{Category}_cluster_montage.png",
            Grid_Cols=4,
        )
else:
    print("No cluster visuals found under", FIGURES_DIR)

[Xrf_Plots] saved category montage (8 images) to c:\Users\gabri\Documento\Mitacs\research_ct\data\xrf\output\comparison\montages\text_only_cluster_montage.png


## Save comparison summary

In [17]:
Tagged_Pages = Category_Registry.List_Tagged_Pages(PROCESSED_DATA_DIR, comparison_config)

Summary = {
    "page_counts": {Category: len(Page_Ids) for Category, Page_Ids in Tagged_Pages.items()},
    "low_confidence_categories": [
        Category
        for Category, Page_Ids in Tagged_Pages.items()
        if Category != "untagged" and len(Page_Ids) < comparison_config.Min_Pages_Per_Category
    ],
    "config": {
        "Allowed_Categories": comparison_config.Allowed_Categories,
        "Min_Pages_Per_Category": comparison_config.Min_Pages_Per_Category,
        "Rarity_Mad_Threshold": comparison_config.Rarity_Mad_Threshold,
        "Min_Region_Size": comparison_config.Min_Region_Size,
    },
}

with open(COMPARISON_DIR / "category_comparison_summary.json", "w") as Summary_File:
    json.dump(Summary, Summary_File, indent=2)

Summary

{'page_counts': {'text_only': 1},
 'low_confidence_categories': ['text_only'],
 'config': {'Allowed_Categories': ['text_only',
   'chapter_start',
   'illustration',
   'mixed',
   'unknown'],
  'Min_Pages_Per_Category': 5,
  'Rarity_Mad_Threshold': 3.5,
  'Min_Region_Size': 10}}